# 3. Kiểm tra và làm sạch dữ liệu

Không cần phần trước.

Muốn chạy toàn bộ trong một kernel: [mở notebook 00](00_Gamer_Behavior_Clustering.ipynb).

<details><summary>Danh sách tám phần</summary>

- [1. Tìm hiểu bài toán và dữ liệu](01_data_understanding.ipynb)

- [2. Thống kê và khám phá dữ liệu (EDA)](02_statistics_eda.ipynb)

- [3. Kiểm tra và làm sạch dữ liệu](03_data_cleaning.ipynb)

- [4. Chia dữ liệu và chuẩn hóa](04_data_transformation.ipynb)

- [5. Ward, K-Means và cách kết hợp](05_clustering.ipynb)

- [6. Đánh giá holdout và độ nhạy](06_model_evaluation.ipynb)

- [7. Hồ sơ người chơi và trực quan hóa](07_player_profiling.ipynb)

- [8. Gán người chơi và đề xuất chiến lược](08_prediction_strategy.ipynb)

</details>

## Khởi tạo và đọc dữ liệu

In [1]:
from pathlib import Path
import os, sys, json, hashlib
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown

# Tìm gốc dự án khi mở từ thư mục notebooks/, gamer_profiling/ hoặc workspace.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "gamer_profiling"]
ROOT = next((p for p in candidates if (p / "gamer_clustering.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án gamer_profiling hoặc notebooks/.")
NBOUT = Path(os.environ.get("GAMER_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebook/latest")))
NBOUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(NBOUT / ".matplotlib"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from gamer_clustering import clean_features, centers_from_labels, metrics, choose_k
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (pairwise_distances, silhouette_score, silhouette_samples,
                             davies_bouldin_score, adjusted_rand_score)
from sklearn.metrics.pairwise import pairwise_distances_argmin
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, cut_tree, dendrogram
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt
import seaborn as sns

# Module .py dùng Agg khi chạy batch; trong notebook cần hiển thị hình ngay dưới ô code.
get_ipython().run_line_magic("matplotlib", "inline")
thread_limit = threadpool_limits(limits=2)
sns.set_theme(style="whitegrid", font="DejaVu Sans", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)
SEED = 42
FEATURES = ["PlayTimeHours", "SessionsPerWeek", "AvgSessionDurationMinutes",
            "PlayerLevel", "AchievementsUnlocked"]
METHODS = ["KMeans", "Ward", "Ward_KMeans"]
SOURCE = ROOT / "data/online_gaming_behavior_dataset.csv"
df = pd.read_csv(SOURCE)
reference = json.loads((ROOT / "outputs/summary.json").read_text(encoding="utf-8"))

def require(name, previous):
    path = NBOUT / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}. Chạy notebook {previous} trước, hoặc Run All notebook 00.")
    return path

def save_plot(name):
    plt.tight_layout()
    plt.savefig(NBOUT / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

Chuỗi trắng, chuỗi không chuyển được thành số, số âm và vô cực phải được kiểm tra trước khi tin vào `isna()`.
Ở đây chỉ chuyển các giá trị không hợp lệ thành thiếu bằng quy tắc xác định;
**chưa tính trung vị trên toàn bộ dữ liệu**. Bộ imputer sẽ được fit sau khi chia train/holdout.

In [2]:
raw = df[FEATURES]
numeric = raw.apply(pd.to_numeric, errors="coerce")
cleaning = pd.DataFrame({
    "missing_raw": raw.isna().sum(),
    "blank_strings": raw.map(lambda v: isinstance(v, str) and v.strip() == "").sum(),
    "invalid_numeric": (numeric.isna() & raw.notna()).sum(),
    "negative": numeric.lt(0).sum(),
    "infinite": pd.Series(np.isinf(numeric).sum(axis=0), index=FEATURES),
})
X = clean_features(df)
cleaning["missing_after_rules"] = X.isna().sum()
display(cleaning)
assert not df.PlayerID.duplicated().any(), "Cần quyết định cách xử lý PlayerID trùng."
assert not df.duplicated().any(), "Cần kiểm tra nguyên nhân trùng dòng."
X.to_csv(NBOUT / "clean_features.csv", index=False, encoding="utf-8-sig")
cleaning.to_csv(NBOUT / "cleaning_audit.csv", encoding="utf-8-sig")
print("Giữ lại", len(X), "người; không xóa dòng tùy tiện.")

,missing_raw,blank_strings,invalid_numeric,negative,infinite,missing_after_rules
PlayTimeHours,0,0,0,0,0,0
SessionsPerWeek,0,0,0,0,0,0
AvgSessionDurationMinutes,0,0,0,0,0,0
PlayerLevel,0,0,0,0,0,0
AchievementsUnlocked,0,0,0,0,0,0


Giữ lại 40034 người; không xóa dòng tùy tiện.


Quy tắc IQR dưới đây chỉ đánh dấu để quan sát, không cắt dữ liệu.
Sau bước làm sạch, mỗi người vẫn giữ nguyên vị trí dòng để truy vết theo `PlayerID`.

In [3]:
q1, q3 = X.quantile(.25), X.quantile(.75)
iqr = q3 - q1
iqr_audit = pd.DataFrame({"q1": q1, "q3": q3,
                          "below_fence": X.lt(q1 - 1.5 * iqr).sum(),
                          "above_fence": X.gt(q3 + 1.5 * iqr).sum()})
display(iqr_audit)
display(X.head())
display(Markdown(f"Sau quy tắc làm sạch có **{int(X.isna().sum().sum())} ô thiếu** "
                 "trong năm đặc trưng. Imputer vẫn được lưu để xử lý đầu vào mới nhất quán."))

,q1,q3,below_fence,above_fence
PlayTimeHours,6.0675,17.9638,0,0
SessionsPerWeek,4.0000,14.0000,0,0
AvgSessionDurationMinutes,52.0000,137.0000,0,0
PlayerLevel,25.0000,74.0000,0,0
AchievementsUnlocked,12.0000,37.0000,0,0


,PlayTimeHours,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked
0,16.2711,6,108,79,25
1,5.5260,5,144,11,10
2,8.2238,16,142,35,41
3,5.2654,9,85,57,47
4,15.5319,2,131,95,37


Sau quy tắc làm sạch có **0 ô thiếu** trong năm đặc trưng. Imputer vẫn được lưu để xử lý đầu vào mới nhất quán.